In [0]:
spark.version

## Creating CUSTOMERS_RAW Table

In [0]:
from pyspark.sql import functions as F

NUM_COSTUMERS = 50_000

In [0]:
# initialize rows data
customers_data = spark.range(1,NUM_COSTUMERS+1)

In [0]:
# adding columns and value data
customers_data = customers_data.withColumns({
    'customer_id' : F.format_string("C%06d", F.col('id')),
    'first_name' : F.element_at(F.array(F.lit('Ali'), F.lit('Marry'), F.lit('Muhammad'), F.lit('John'), F.lit('Abishek'), F.lit('Khairul'), F.lit('Zyan'), F.lit('Lucy'), F.lit('Chika'), F.lit('Gerrard')), (F.rand() * 10 + 1).cast('int')),
    'last_name' : F.element_at(F.array(F.lit('Zamir'), F.lit('Catherina'), F.lit('Abdullah'), F.lit('Smith'), F.lit('Rajesh'), F.lit('Khamis'), F.lit('Ryan'), F.lit('Lisa'), F.lit('Chota'), F.lit('Michael')), (F.rand() * 10 + 1).cast('int')),
    'email': F.lower(F.concat(F.col('first_name'), F.lit('.'), F.col('last_name'), F.lit('@nocodeah.com'))),
    'country': F.element_at(F.array(F.lit('Malaysia'), F.lit('UK'), F.lit('Germany'), F.lit('Australia'), F.lit('India'), F.lit('China'), F.lit('Japan')), (F.rand() * 7 + 1).cast('int')),
    'customer_segment': F.element_at(F.array(F.lit('Standard'), F.lit('Premium'), F.lit('Enterprise')), (F.rand() * 3 + 1).cast('int')),
    'signup_date': F.date_sub(F.current_date(),(F.rand() * 1000).cast("int"))
}).drop('id')

In [0]:
customers_data.count()
customers_data.printSchema()

In [0]:
customers_data.write\
    .format('delta')\
    .mode('overwrite')\
    .saveAsTable('customers_raw')

In [0]:
display(spark.sql('SHOW TABLES').show())
display(spark.table("customers_raw").show(10))

## Creating PRODUCTS_RAW Table

In [0]:
# intialize rows data
NUM_PRODUCTS = 5000
products_data = spark.range(1, NUM_PRODUCTS+1)

# adding columns and value data
products_data = products_data.withColumns({
    'product_id': F.format_string('P%06d', F.col('id')),
    'category' : F.element_at(F.array(F.lit("Electronics"),F.lit("Home"),F.lit("Fashion"),F.lit("Sports"),F.lit("Beauty")), (F.rand()*5+1).cast('int')),
    'subacategory':
        F.when(F.col('category')=='Electronics', 'Accessories')
        .when(F.col("category") == "Home", "Kitchen")
        .when(F.col("category") == "Fashion", "Clothing")
        .when(F.col("category") == "Sports", "Fitness")
        .otherwise("Personal Care"),
    'product_name' : F.concat(F.col("category"),F.lit(" Product "),F.col("product_id")),
    'unit_cost' : F.round(F.rand() * 200 + 5, 2),
    'unit_price' : F.round(F.col("unit_cost") * (F.rand() * 2 + 1.2),2)
}).drop('id')


In [0]:
display(products_data.groupBy('category').count().show())
display(products_data.head(10))

# check invalid product data, if unit_cost > unit_price
display(products_data.filter(F.col('unit_cost') > F.col('unit_price')).count())

In [0]:
products_data.write.format('delta').mode('overwrite').saveAsTable('products_raw')

## Creating ORDER Table

In [0]:
NUM_ORDERS=500_000
orders = spark.range(1, NUM_ORDERS+1)\
    .withColumns({
        'order_id' : F.format_string('O%08d', F.col('id')),
        'customer_id': F.format_string('C%06d', (F.rand()*50_000+1).cast('int')),
        'order_date': F.date_sub(F.current_date(), (F.rand() * 365).cast('int')),
        'order_status': F.element_at(F.array(F.lit('COMPLETED'), F.lit('COMPLETED'), F.lit('COMPLETED'), F.lit('PENDING'), F.lit('CANCELLED')), (F.rand() * 5 + 1).cast('int')),
        'payment_method': F.element_at(F.array(F.lit("CREDIT_CARD"),F.lit("DEBIT_CARD"),F.lit("BANK_TRANSFER"),F.lit("E_WALLET")),(F.rand()* 4 + 1).cast('int'))
    })\
    .drop('id')

In [0]:
orders.write.format('delta').mode('overwrite').saveAsTable('orders_raw')

## Creating ORDER_ITEM Table

In [0]:
NUM_ORDER_ITEMS = 1_000_000
order_items = spark.range(1,NUM_ORDER_ITEMS+1)\
    .withColumns({
        'order_item_id' : F.format_string("OI%09d", F.col("id")),
        'order_id' : F.format_string('O%08d', (F.rand() * NUM_ORDERS +1).cast('int')),
        'product_id' : F.format_string('P%06d', (F.rand() * NUM_PRODUCTS + 1).cast('int')),
        'quantity' : (F.rand()*5+1).cast('int'),
        'discount' : F.round(F.rand()*0.2,2)
    })\
    .drop('id')

In [0]:
# joining product price
order_items = order_items.join(
    products_data.select('product_id','unit_price'), 
    on='product_id', 
    how='left'
    )

In [0]:
order_items = (order_items
 .withColumn('gross_amount', F.round(F.col('quantity') * F.col('unit_price'), 2))
 .withColumn('discount_amount', F.round(F.col('gross_amount') * F.col('discount'),2))
 .withColumn('net_amount', F.round(F.col('gross_amount') - F.col('discount_amount'),2))
 )

In [0]:
(order_items.write
 .format('delta')
 .mode('overwrite')
 .option('overwriteSchema', 'true')
 .saveAsTable('order_items_raw'))

In [0]:
NUM_PRODUCTS=5_000

# Creating Payments Table

In [0]:
from pyspark.sql import functions as F

orders = spark.sql('select * from orders_raw')
order_items = spark.sql('select * from order_items_raw')

In [0]:
payments = (
    orders
    .select(
        "order_id",
        "order_date",
        "payment_method",
        "order_status"
    )
    .withColumn(
        "payment_id",
        F.concat(
            F.lit("PAY"),
            F.regexp_replace("order_id", "O", "")
        )
    )
    .withColumn(
        "payment_date",
        F.when(
            F.col("order_status") == "COMPLETED",
            F.col("order_date")
        ).otherwise(
            F.date_add(
                F.col("order_date"),
                (F.rand() * 3).cast("int")
            )
        )
    )
    .withColumn(
        "payment_status",
        F.when(
            F.col("order_status") == "COMPLETED",
            F.lit("PAID")
        )
        .when(
            F.col("order_status") == "PENDING",
            F.lit("PENDING")
        )
        .otherwise(
            F.lit("FAILED")
        )
    )
    .select(
        "payment_id",
        "order_id",
        "payment_date",
        "payment_method",
        "payment_status"
    )
)

In [0]:
order_totals = (
    order_items
    .groupBy('order_id')
    .agg(
        F.round(F.sum('net_amount'), 2).alias('order_amount')
        )
)

In [0]:
payments = (
    payments
    .join(order_totals, on='order_id', how='left')
)

In [0]:
payments = (
    payments
    .withColumn(
        'amount',
        F.when(
            F.col("payment_status") == "PAID",
            F.col("order_amount")
        )
        .otherwise(
            F.lit(0.0)
        )
    )
    .drop('order_amount')
)


reconciliation = (
    orders
    .join(order_totals, on='order_id', how='left')
    .join(payments.select('order_id','payment_status','amount'), on='order_id', how='left')
    .withColumn('difference', F.round(F.col('order_amount')-F.col('amount'),2))
)

In [0]:
reconciliation\
.filter(F.col('order_status') == 'COMPLETED')\
.select('order_id','order_amount','amount','difference')\
.show(10)

(
    payments
    .join(
        orders.select("order_id"),
        on="order_id",
        how="left_anti"
    )
).count()

reconciliation.filter(
    (F.col('order_status') == 'COMPLETED') & (F.abs(F.col('difference')) > 0.01) 
).count()

In [0]:
payments.write.format('delta').mode('overwrite').saveAsTable('payments_raw')

In [0]:
spark.sql('show tables').show()

## Data Exploratory

In [0]:
spark.sql('select * from products_raw fetch limit 10').show()

In [0]:
spark.sql("""
    SELECT
        category,
        COUNT(*) AS product_count,
        ROUND(AVG(unit_price), 2) AS avg_price
    FROM products_raw
    GROUP BY category
    ORDER BY product_count DESC
""").show()

In [0]:
customers_data = spark.read.table('customers_raw')

In [0]:
# orders.join(customers_data, how='left_anti', on="customer_id").count()
    # .filter((customers_data.country == 'Malaysia') & (customers_data.customer_id == 'C006547')).show(10)

spark.sql('show tables').show()

orders.select(
    F.max(orders.order_date).alias("max_order_date"),
    F.min(orders.order_date).alias("min_order_date")
).show()

In [0]:
# spark.sql("""select c.customer_segment, count(*) from customers_raw as c 
#           inner join orders_raw as o on c.customer_id=o.customer_id
#           group by c.customer_segment order by count(*)
#           """).show(5)

customers_data.join(orders, on="customer_id", how='inner')\
    .groupBy('customer_segment').count()\
        .orderBy('count').show(10)

In [0]:
spark.sql('show tables').show()
spark.sql('desc customers_raw').show()
spark.sql('desc products_raw').show()
spark.sql('desc orders_raw').show()
spark.sql('desc order_items_raw').show()

In [0]:
products_data = spark.read.table('products_raw')
invalid_products = (
    order_items
    .join(
        products.select("product_id"),
        on="product_id",
        how="left_anti"
    )
)
invalid_products.count()

In [0]:
spark.sql("""
    SELECT
        ROUND(SUM(net_amount), 2) AS total_revenue,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(
            SUM(net_amount) /
            COUNT(DISTINCT order_id),
            2
        ) AS average_order_value
    FROM order_items_raw
""").show()